# Building a Complete RAG Pipeline

In this notebook we build a fully functional RAG (Retrieval-Augmented Generation) pipeline from scratch. The pipeline has four stages:

1. **Knowledge Base** — A set of policy documents for a fictional company called *Acme Corp*.
2. **Embedding & Indexing** — Documents are chunked, embedded with OpenAI, and stored in Qdrant.
3. **Retrieval & Reranking** — Given a query, we retrieve relevant chunks and optionally rerank with a cross-encoder model.
4. **Generation** — The retrieved context is fed to GPT-4o-mini to produce an answer.

By the end you will have a `rag_pipeline(query)` function that returns `(answer, retrieved_contexts)` — the exact format needed for evaluation in Notebooks 03-05.

-
## 1. Setup & Imports

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
if not os.path.exists(dotenv_path):
    dotenv_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path, override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json
import hashlib
from typing import List, Tuple, Dict, Optional

print("All imports successful.")

All imports successful.


-
## 2. Knowledge Base — Acme Corp Policy Documents

We create 15 detailed policy documents for a fictional e-commerce company called **Acme Corp**. These cover returns, shipping, products, warranties, support, and more. Each document is a plain string simulating a page from the company knowledge base.

In [3]:
KNOWLEDGE_BASE = [
    # Document 1 — Return Policy
    (
        "Return Policy Overview",
        "Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. "
        "To be eligible for a return, the item must be unused, in its original packaging, and accompanied by the "
        "original receipt or proof of purchase. Refunds are processed within 5-7 business days after we receive "
        "the returned item. Shipping costs for returns are the responsibility of the customer unless the item "
        "was defective or the wrong item was shipped. Items marked as 'Final Sale' cannot be returned or exchanged."
    ),
    # Document 2 — Extended Return for Electronics
    (
        "Electronics Return Policy",
        "Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days. "
        "All electronics must be returned with all original accessories, cables, manuals, and packaging. A restocking "
        "fee of 15% may apply to opened electronics. Defective electronics can be exchanged for the same model within "
        "the first 90 days of purchase at no additional cost. Software products and digital downloads are non-returnable "
        "once the seal is broken or the download code has been redeemed."
    ),
    # Document 3 — Shipping Policy
    (
        "Shipping Policy",
        "Acme Corp offers several shipping options for domestic orders within the United States. Standard Shipping "
        "takes 5-7 business days and is free for orders over $50. Expedited Shipping takes 2-3 business days and "
        "costs $12.99. Overnight Shipping is available for $24.99 and delivers the next business day if ordered "
        "before 2 PM EST. Orders are processed within 1-2 business days. Tracking information is sent via email "
        "once the package has been shipped. Acme Corp is not responsible for delays caused by weather, carrier "
        "issues, or incorrect shipping addresses provided by the customer."
    ),
    # Document 4 — International Shipping
    (
        "International Shipping Policy",
        "Acme Corp ships internationally to over 50 countries. International Standard Shipping takes 10-21 business "
        "days and costs vary by destination. International Express Shipping takes 5-7 business days. All international "
        "orders may be subject to customs duties, taxes, and import fees which are the responsibility of the customer. "
        "Acme Corp declares the full value of items on customs forms. We do not mark packages as gifts or undervalue "
        "them for customs purposes. International returns must be shipped at the customer's expense, and refunds will "
        "not include the original shipping cost."
    ),
    # Document 5 — Product Warranty
    (
        "Product Warranty Information",
        "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and "
        "workmanship under normal use. The warranty does not cover damage caused by accidents, misuse, unauthorized "
        "modifications, or normal wear and tear. To make a warranty claim, customers must provide proof of purchase "
        "and a description of the defect. Warranty service may include repair, replacement, or refund at Acme Corp's "
        "discretion. Extended warranty plans (2-year and 3-year) are available for purchase at the time of original "
        "purchase for an additional fee."
    ),
    # Document 6 — Customer Support
    (
        "Customer Support Channels",
        "Acme Corp customer support is available through multiple channels. Phone support is available Monday through "
        "Friday, 8 AM to 8 PM EST at 1-800-ACME-HELP (1-800-226-3435). Email support can be reached at "
        "support@acmecorp.com with a typical response time of 24-48 hours. Live chat is available on our website "
        "Monday through Saturday, 9 AM to 6 PM EST. For urgent issues outside business hours, customers can use our "
        "automated help center at help.acmecorp.com which includes troubleshooting guides, FAQs, and order tracking."
    ),
    # Document 7 — Product: Acme SmartHome Hub
    (
        "Product: Acme SmartHome Hub",
        "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99. It supports WiFi, "
        "Bluetooth, Zigbee, and Z-Wave protocols, making it compatible with over 10,000 smart home devices. Features "
        "include voice control via built-in microphones, a 5-inch touchscreen display, energy monitoring dashboard, "
        "and automated routines. The hub requires a stable WiFi connection (2.4 GHz or 5 GHz) and can be set up "
        "using the Acme Home app available for iOS and Android. The SmartHome Hub comes with a 2-year warranty."
    ),
    # Document 8 — Product: Acme AirPure Pro
    (
        "Product: Acme AirPure Pro Air Purifier",
        "The Acme AirPure Pro is a premium air purifier designed for rooms up to 800 square feet, priced at $299.99. "
        "It features a 4-stage filtration system: pre-filter, activated carbon filter, True HEPA H13 filter, and "
        "UV-C light sanitizer. The AirPure Pro removes 99.97% of particles as small as 0.3 microns including dust, "
        "pollen, pet dander, mold spores, and smoke. It operates at noise levels as low as 24 dB on sleep mode. "
        "Filter replacements are recommended every 6-8 months and cost $49.99 for the HEPA filter and $29.99 for "
        "the carbon filter. The unit comes with a 1-year warranty."
    ),
    # Document 9 — Product: Acme FitBand Ultra
    (
        "Product: Acme FitBand Ultra Fitness Tracker",
        "The Acme FitBand Ultra is a fitness tracker priced at $79.99 that monitors heart rate, steps, sleep quality, "
        "blood oxygen levels, and stress. It features a 1.4-inch AMOLED display, 7-day battery life, water resistance "
        "up to 50 meters (5 ATM), and GPS tracking for outdoor workouts. The FitBand Ultra syncs with the Acme Health "
        "app to provide detailed analytics and personalized health insights. It is compatible with both iOS and Android. "
        "Band accessories are available in silicone, leather, and stainless steel. The FitBand Ultra includes a 1-year "
        "warranty covering manufacturing defects."
    ),
    # Document 10 — Payment Methods
    (
        "Accepted Payment Methods",
        "Acme Corp accepts the following payment methods: Visa, MasterCard, American Express, Discover, PayPal, "
        "Apple Pay, Google Pay, and Acme Gift Cards. For orders over $200, we also offer Acme Pay Later, a buy-now-pay-later "
        "option that splits payments into 4 interest-free installments over 6 weeks. All payment information is encrypted "
        "using 256-bit SSL encryption. We do not store credit card numbers on our servers. For business accounts, we also "
        "accept purchase orders (PO) with Net-30 terms for approved accounts."
    ),
    # Document 11 — Loyalty Program
    (
        "Acme Rewards Loyalty Program",
        "The Acme Rewards program is free to join and earns members 1 point per dollar spent. Points can be redeemed "
        "for discounts: 100 points equals $5 off your next purchase. Members also receive exclusive benefits including "
        "early access to sales, birthday discounts (20% off), and free standard shipping on all orders. Silver tier "
        "(500+ points/year) unlocks free expedited shipping. Gold tier (1000+ points/year) adds priority customer "
        "support and exclusive product previews. Points expire after 12 months of account inactivity."
    ),
    # Document 12 — Privacy Policy
    (
        "Privacy Policy Summary",
        "Acme Corp collects personal information including name, email, shipping address, payment information, and "
        "browsing behavior to process orders and improve our services. We do not sell personal information to third "
        "parties. Data may be shared with shipping carriers, payment processors, and analytics providers as necessary "
        "to fulfill orders. Customers can request data deletion by contacting privacy@acmecorp.com. We comply with "
        "GDPR, CCPA, and other applicable data protection regulations. Cookies can be managed through browser settings "
        "or our cookie preference center on the website."
    ),
    # Document 13 — Product: Acme ErgoDesk Pro
    (
        "Product: Acme ErgoDesk Pro Standing Desk",
        "The Acme ErgoDesk Pro is a motorized standing desk priced at $599.99. It features a 60x30 inch bamboo "
        "desktop, dual-motor height adjustment from 25.5 to 51 inches, 4 programmable height presets, built-in cable "
        "management tray, and anti-collision technology. The desk supports up to 300 lbs and adjusts at a speed of "
        "1.5 inches per second. Assembly takes approximately 30 minutes with two people. The ErgoDesk Pro comes with "
        "a 5-year warranty on the frame and motors, and a 2-year warranty on the electronics and desktop surface."
    ),
    # Document 14 — Order Cancellation
    (
        "Order Cancellation Policy",
        "Orders can be cancelled within 1 hour of placement by contacting customer support or using the 'Cancel Order' "
        "button in your account dashboard. After 1 hour, orders enter the processing queue and may not be cancellable. "
        "If an order has already shipped, it cannot be cancelled and must be returned using our standard return process. "
        "Custom or personalized items cannot be cancelled once production has begun. Cancelled orders are refunded to "
        "the original payment method within 3-5 business days."
    ),
    # Document 15 — Product: Acme CloudStorage Plan
    (
        "Product: Acme CloudStorage Plans",
        "Acme CloudStorage offers secure cloud storage for photos, documents, and files. Plans include: Basic (50 GB) "
        "at $2.99/month, Standard (200 GB) at $5.99/month, and Premium (2 TB) at $12.99/month. All plans include "
        "end-to-end encryption, automatic backup, file versioning (last 30 days), and sharing with up to 5 family "
        "members on Premium. Annual subscriptions save 20%. Acme CloudStorage apps are available for Windows, macOS, "
        "iOS, and Android. Business plans with admin controls and audit logs start at $8.99/user/month."
    ),
]

print(f"Knowledge base contains {len(KNOWLEDGE_BASE)} documents:")
for i, (title, _) in enumerate(KNOWLEDGE_BASE, 1):
    print(f"  {i:2d}. {title}")

Knowledge base contains 15 documents:
   1. Return Policy Overview
   2. Electronics Return Policy
   3. Shipping Policy
   4. International Shipping Policy
   5. Product Warranty Information
   6. Customer Support Channels
   7. Product: Acme SmartHome Hub
   8. Product: Acme AirPure Pro Air Purifier
   9. Product: Acme FitBand Ultra Fitness Tracker
  10. Accepted Payment Methods
  11. Acme Rewards Loyalty Program
  12. Privacy Policy Summary
  13. Product: Acme ErgoDesk Pro Standing Desk
  14. Order Cancellation Policy
  15. Product: Acme CloudStorage Plans


-
## 3. Document Chunking

Even though our documents are relatively short, real-world RAG systems need chunking. We implement a simple sentence-level chunker with overlap to demonstrate the concept.

In [4]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Split text into chunks of approximately `chunk_size` characters
    with `overlap` characters of overlap between consecutive chunks.
    Splits on sentence boundaries where possible.
    """
    sentences = text.replace(". ", ".\n").split("\n")
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current_chunk) + len(sentence) + 1 <= chunk_size:
            current_chunk = (current_chunk + " " + sentence).strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            # Start new chunk with overlap from end of previous
            if overlap > 0 and current_chunk:
                overlap_text = current_chunk[-overlap:]
                current_chunk = overlap_text + " " + sentence
            else:
                current_chunk = sentence
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

# Test chunking on one document
test_chunks = chunk_text(KNOWLEDGE_BASE[0][1])
print(f"Document '{KNOWLEDGE_BASE[0][0]}' split into {len(test_chunks)} chunk(s):")
for i, chunk in enumerate(test_chunks):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

Document 'Return Policy Overview' split into 3 chunk(s):
  Chunk 1 (246 chars): Acme Corp offers a 30-day return policy on all products purchased through our we...
  Chunk 2 (260 chars): nied by the original receipt or proof of purchase. Refunds are processed within ...
  Chunk 3 (111 chars): item was defective or the wrong item was shipped. Items marked as 'Final Sale' c...


In [5]:
# Chunk all documents and store metadata
all_chunks = []  # list of dicts: {"id", "text", "title", "doc_index"}

for doc_index, (title, text) in enumerate(KNOWLEDGE_BASE):
    chunks = chunk_text(text, chunk_size=400, overlap=50)
    for chunk_index, chunk in enumerate(chunks):
        chunk_id = hashlib.md5(f"{title}_{chunk_index}".encode()).hexdigest()
        all_chunks.append({
            "id": chunk_id,
            "text": chunk,
            "title": title,
            "doc_index": doc_index,
            "chunk_index": chunk_index,
        })

print(f"Total chunks created: {len(all_chunks)}")
for c in all_chunks:
    print(f"  [{c['doc_index']:2d}-{c['chunk_index']}] {c['title'][:40]:<40s} ({len(c['text']):3d} chars)")

Total chunks created: 30
  [ 0-0] Return Policy Overview                   (329 chars)
  [ 0-1] Return Policy Overview                   (238 chars)
  [ 1-0] Electronics Return Policy                (376 chars)
  [ 1-1] Electronics Return Policy                (173 chars)
  [ 2-0] Shipping Policy                          (376 chars)
  [ 2-1] Shipping Policy                          (257 chars)
  [ 3-0] International Shipping Policy            (387 chars)
  [ 3-1] International Shipping Policy            (246 chars)
  [ 4-0] Product Warranty Information             (347 chars)
  [ 4-1] Product Warranty Information             (264 chars)
  [ 5-0] Customer Support Channels                (349 chars)
  [ 5-1] Customer Support Channels                (222 chars)
  [ 6-0] Product: Acme SmartHome Hub              (341 chars)
  [ 6-1] Product: Acme SmartHome Hub              (232 chars)
  [ 7-0] Product: Acme AirPure Pro Air Purifier   (361 chars)
  [ 7-1] Product: Acme AirPure Pro Air Purifi

-
## 4. Generate Embeddings with OpenAI

We use OpenAI's `text-embedding-3-small` model (1536 dimensions) to embed each chunk.

In [6]:
openai_client = OpenAI()

def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """
    Get embeddings for a list of texts using OpenAI API.
    Handles batching automatically.
    """
    response = openai_client.embeddings.create(
        model=model,
        input=texts,
    )
    return [item.embedding for item in response.data]

# Embed all chunks
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print(f"Generated {len(chunk_embeddings)} embeddings")
print(f"Embedding dimension: {len(chunk_embeddings[0])}")

Generated 30 embeddings
Embedding dimension: 1536


-
## 5. Set Up Qdrant Vector Store

We create an in-memory Qdrant collection and ingest all chunks with their embeddings.

In [7]:
COLLECTION_NAME = "acme_corp_kb"
EMBEDDING_DIM = 1536

# Create in-memory Qdrant client
qdrant_client = QdrantClient(":memory:")

# Create collection
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE,
    ),
)
print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'acme_corp_kb' created.


In [8]:
# Upsert all chunks with embeddings
points = []
for i, (chunk, embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": chunk["text"],
                "title": chunk["title"],
                "doc_index": chunk["doc_index"],
                "chunk_index": chunk["chunk_index"],
            },
        )
    )

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

collection_info = qdrant_client.get_collection(COLLECTION_NAME)
print(f"Upserted {collection_info.points_count} points into '{COLLECTION_NAME}'.")

Upserted 30 points into 'acme_corp_kb'.


-
## 6. Implement Retriever Function

The retriever takes a query, embeds it, and searches Qdrant for the most similar chunks.

In [9]:
def retrieve(query: str, top_k: int = 5) -> List[Dict]:
    """
    Retrieve the top-k most relevant chunks for a given query.
    
    Returns:
        List of dicts with keys: text, title, score
    """
    query_embedding = get_embeddings([query])[0]
    
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k,
        with_payload=True,
    )
    
    retrieved = []
    for point in results.points:
        retrieved.append({
            "text": point.payload["text"],
            "title": point.payload["title"],
            "score": point.score,
        })
    
    return retrieved

# Test the retriever
test_results = retrieve("What is your return policy?")
print("Query: 'What is your return policy?'\n")
for i, r in enumerate(test_results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['title']}")
    print(f"     {r['text'][:100]}...\n")

Query: 'What is your return policy?'

  1. [0.5546] Return Policy Overview
     Acme Corp offers a 30-day return policy on all products purchased through our website or retail stor...

  2. [0.5171] Return Policy Overview
     business days after we receive the returned item. Shipping costs for returns are the responsibility ...

  3. [0.5038] Electronics Return Policy
     Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 ...

  4. [0.4915] Order Cancellation Policy
     ust be returned using our standard return process. Custom or personalized items cannot be cancelled ...

  5. [0.4607] Electronics Return Policy
     e first 90 days of purchase at no additional cost. Software products and digital downloads are non-r...



-
## 7. Implement Reranker (Cross-Encoder)

Reranking takes the initial retrieval results and reorders them using a cross-encoder model. This typically improves precision. We use the `cross-encoder/ms-marco-MiniLM-L-6-v2` model from sentence-transformers, which runs locally — no API key required.

In [10]:
from sentence_transformers import CrossEncoder

# Load the cross-encoder model (runs locally, no API key needed)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, documents: List[Dict], top_n: int = 3) -> List[Dict]:
    """
    Rerank retrieved documents using a cross-encoder model.
    
    Args:
        query: The search query
        documents: List of dicts with 'text' key
        top_n: Number of documents to return after reranking
    
    Returns:
        Reranked list of document dicts (highest relevance first)
    """
    if not documents:
        return []

    pairs = [[query, d["text"]] for d in documents]
    scores = cross_encoder.predict(pairs)

    scored_docs = []
    for idx, score in enumerate(scores):
        doc = documents[idx].copy()
        doc["rerank_score"] = float(score)
        scored_docs.append(doc)

    scored_docs.sort(key=lambda d: d["rerank_score"], reverse=True)
    print(f"  [Cross-encoder reranker applied — {min(top_n, len(scored_docs))} results]")
    return scored_docs[:top_n]


# Test the reranker
retrieved = retrieve("How do I return an electronic item?", top_k=5)
reranked = rerank("How do I return an electronic item?", retrieved, top_n=3)

print("\nReranked results:")
for i, r in enumerate(reranked, 1):
    score_info = f"rerank={r['rerank_score']:.4f}" if "rerank_score" in r else f"cosine={r['score']:.4f}"
    print(f"  {i}. [{score_info}] {r['title']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Cross-encoder reranker applied — 3 results]

Reranked results:
  1. [rerank=4.9943] Electronics Return Policy
  2. [rerank=-0.4838] Return Policy Overview
  3. [rerank=-1.8744] Return Policy Overview


-
## 8. Implement Generator Function

The generator takes a query and a list of context documents, then uses GPT-4o-mini to produce an answer. We use a carefully crafted system prompt to encourage grounded, faithful responses.

In [11]:
SYSTEM_PROMPT = """You are a helpful customer support assistant for Acme Corp.
Answer the customer's question based ONLY on the provided context documents.
If the context does not contain enough information to answer the question, say so clearly.
Do not make up information that is not supported by the context.
Be concise but thorough in your response."""


def generate(query: str, contexts: List[Dict], model: str = "gpt-4o-mini") -> str:
    """
    Generate an answer to the query using the provided contexts.
    
    Args:
        query: The user's question
        contexts: List of dicts with 'text' and 'title' keys
        model: OpenAI model to use
    
    Returns:
        Generated answer string
    """
    # Build context string
    context_str = "\n\n".join(
        f"[Source: {c['title']}]\n{c['text']}" for c in contexts
    )
    
    user_message = f"""Context Documents:
---
{context_str}
---

Customer Question: {query}

Please provide a helpful answer based on the context above."""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    
    return response.choices[0].message.content

# Test the generator
test_contexts = retrieve("What is your return policy?", top_k=3)
test_answer = generate("What is your return policy?", test_contexts)

print("Query: 'What is your return policy?'\n")
print(f"Answer:\n{test_answer}")

Query: 'What is your return policy?'

Answer:
Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. To be eligible for a return, the item must be unused, in its original packaging, and accompanied by the original receipt or proof of purchase. Refunds are processed within 5-7 business days after we receive the returned item. 

For electronic products, there is a 15-day return window, and all electronics must be returned with all original accessories, cables, manuals, and packaging. A restocking fee of 15% may apply to opened electronics. Defective electronics can be exchanged for the same model within the first 90 days of purchase at no additional cost. 

Please note that shipping costs for returns are the responsibility of the customer unless the item was defective or the wrong item was shipped, and items marked as 'Final Sale' cannot be returned or exchanged.


-
## 9. Build Complete RAG Pipeline

Now we combine retrieval, reranking, and generation into a single `rag_pipeline` function that returns both the answer and the retrieved contexts (needed for evaluation).

In [12]:
def rag_pipeline(
    query: str,
    top_k_retrieve: int = 5,
    top_n_rerank: int = 3,
    use_reranker: bool = True,
    model: str = "gpt-4o-mini",
    verbose: bool = True,
) -> Tuple[str, List[str]]:
    """
    Complete RAG pipeline: Retrieve -> Rerank -> Generate.
    
    Args:
        query: The user's question
        top_k_retrieve: Number of chunks to retrieve initially
        top_n_rerank: Number of chunks to keep after reranking
        use_reranker: Whether to apply cross-encoder reranking
        model: OpenAI model for generation
        verbose: Whether to print progress
    
    Returns:
        Tuple of (answer_string, list_of_context_strings)
    """
    if verbose:
        print(f"Query: {query}")
    
    # Step 1: Retrieve
    if verbose:
        print(f"  Step 1: Retrieving top-{top_k_retrieve} chunks...")
    retrieved = retrieve(query, top_k=top_k_retrieve)
    
    # Step 2: Rerank (optional)
    if use_reranker:
        if verbose:
            print(f"  Step 2: Reranking to top-{top_n_rerank}...")
        contexts = rerank(query, retrieved, top_n=top_n_rerank)
    else:
        contexts = retrieved[:top_n_rerank]
        if verbose:
            print(f"  Step 2: Skipping reranker, using top-{top_n_rerank} by cosine sim.")
    
    # Step 3: Generate
    if verbose:
        print(f"  Step 3: Generating answer with {model}...")
    answer = generate(query, contexts, model=model)
    
    # Extract just the text strings for evaluation
    context_strings = [c["text"] for c in contexts]
    
    if verbose:
        print(f"  Done. Answer length: {len(answer)} chars, Contexts: {len(context_strings)}")
    
    return answer, context_strings

print("rag_pipeline function defined successfully.")

rag_pipeline function defined successfully.


-
## 10. Test the Pipeline End-to-End

Let's run the full pipeline on several sample queries to make sure everything works.

In [13]:
sample_queries = [
    "What is your return policy for electronics?",
    "How much does overnight shipping cost?",
    "Tell me about the SmartHome Hub features.",
    "What payment methods do you accept?",
    "How do I cancel an order that I just placed?",
    "What are the benefits of the Gold loyalty tier?",
    "Does the AirPure Pro work for large rooms?",
]

results = []

for query in sample_queries:
    print("=" * 70)
    answer, contexts = rag_pipeline(query, verbose=True)
    results.append({
        "query": query,
        "answer": answer,
        "contexts": contexts,
    })
    print(f"\n  ANSWER: {answer[:200]}{'...' if len(answer) > 200 else ''}\n")

Query: What is your return policy for electronics?
  Step 1: Retrieving top-5 chunks...
  Step 2: Reranking to top-3...
  [Cross-encoder reranker applied — 3 results]
  Step 3: Generating answer with gpt-4o-mini...
  Done. Answer length: 383 chars, Contexts: 3

  ANSWER: The return policy for electronics at Acme Corp allows for a 15-day return window. All electronic products must be returned with all original accessories, cables, manuals, and packaging. If the electro...

Query: How much does overnight shipping cost?
  Step 1: Retrieving top-5 chunks...
  Step 2: Reranking to top-3...
  [Cross-encoder reranker applied — 3 results]
  Step 3: Generating answer with gpt-4o-mini...
  Done. Answer length: 59 chars, Contexts: 3

  ANSWER: Overnight Shipping costs $24.99 if ordered before 2 PM EST.

Query: Tell me about the SmartHome Hub features.
  Step 1: Retrieving top-5 chunks...
  Step 2: Reranking to top-3...
  [Cross-encoder reranker applied — 3 results]
  Step 3: Generating answer wit

-
## 11. Inspect a Single Result in Detail

Let's look at one result closely to understand what the evaluator will see.

In [14]:
sample = results[0]

print("QUERY:")
print(f"  {sample['query']}\n")

print("RETRIEVED CONTEXTS:")
for i, ctx in enumerate(sample['contexts'], 1):
    print(f"  Context {i}:")
    print(f"    {ctx[:150]}...\n")

print("GENERATED ANSWER:")
print(f"  {sample['answer']}")

QUERY:
  What is your return policy for electronics?

RETRIEVED CONTEXTS:
  Context 1:
    Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days. All electronics must be returned with all or...

  Context 2:
    Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. To be eligible for a return, the item must be ...

  Context 3:
    ust be returned using our standard return process. Custom or personalized items cannot be cancelled once production has begun. Cancelled orders are re...

GENERATED ANSWER:
  The return policy for electronics at Acme Corp allows for a 15-day return window. All electronic products must be returned with all original accessories, cables, manuals, and packaging. If the electronics are opened, a restocking fee of 15% may apply. Additionally, defective electronics can be exchanged for the same model within the first 90 days of purchase at no additional cost.


-
## 12. Create Test Queries with Expected Outputs

For evaluation notebooks (03-05), we need test cases with `expected_output` (ground truth). Let's create these now.

In [15]:
TEST_CASES = [
    {
        "query": "What is the return policy for regular items?",
        "expected_output": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds are processed in 5-7 business days. Shipping costs are the customer's responsibility unless the item was defective.",
    },
    {
        "query": "How long do I have to return electronics?",
        "expected_output": "Electronics have a 15-day return window. They must be returned with all original accessories and packaging. A 15% restocking fee may apply to opened items. Defective electronics can be exchanged within 90 days.",
    },
    {
        "query": "What shipping options are available and how much do they cost?",
        "expected_output": "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered before 2 PM EST).",
    },
    {
        "query": "Do you ship internationally?",
        "expected_output": "Yes, Acme Corp ships to over 50 countries. Standard international shipping takes 10-21 business days, Express takes 5-7 days. Customers are responsible for customs duties and import fees.",
    },
    {
        "query": "What does the warranty cover on Acme products?",
        "expected_output": "Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship. It does not cover accidents, misuse, or normal wear. Extended 2-year and 3-year plans are available.",
    },
    {
        "query": "How can I contact customer support?",
        "expected_output": "Customer support is available by phone (1-800-ACME-HELP, M-F 8AM-8PM EST), email (support@acmecorp.com, 24-48hr response), and live chat (M-Sat 9AM-6PM EST). An automated help center is at help.acmecorp.com.",
    },
    {
        "query": "What are the features of the Acme SmartHome Hub?",
        "expected_output": "The SmartHome Hub costs $149.99, supports WiFi/Bluetooth/Zigbee/Z-Wave, has voice control, 5-inch touchscreen, energy monitoring, and automated routines. It works with 10,000+ devices and comes with a 2-year warranty.",
    },
    {
        "query": "How much is the AirPure Pro and what does it filter?",
        "expected_output": "The AirPure Pro costs $299.99 and uses a 4-stage filtration system (pre-filter, carbon, True HEPA H13, UV-C). It removes 99.97% of particles 0.3 microns or larger, covers rooms up to 800 sq ft.",
    },
    {
        "query": "What payment methods do you accept?",
        "expected_output": "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. Orders over $200 qualify for Acme Pay Later (4 installments over 6 weeks).",
    },
    {
        "query": "How does the Acme Rewards program work?",
        "expected_output": "Acme Rewards is free, earning 1 point per dollar. 100 points = $5 off. Benefits include early sale access, birthday 20% discount, and free shipping. Silver tier (500+ pts) adds free expedited shipping. Gold tier (1000+ pts) adds priority support.",
    },
    {
        "query": "Can I cancel an order after placing it?",
        "expected_output": "Orders can be cancelled within 1 hour via customer support or the account dashboard. After 1 hour, orders may not be cancellable. Shipped orders must go through the return process. Custom items cannot be cancelled once production begins.",
    },
    {
        "query": "Tell me about the Acme ErgoDesk Pro standing desk.",
        "expected_output": "The ErgoDesk Pro costs $599.99, has a 60x30 inch bamboo top, dual-motor height adjustment (25.5-51 inches), 4 presets, cable management, and anti-collision tech. It supports 300 lbs and has a 5-year frame/motor warranty.",
    },
]

print(f"Created {len(TEST_CASES)} test cases for evaluation.")
for i, tc in enumerate(TEST_CASES, 1):
    print(f"  {i:2d}. {tc['query'][:60]}...")

Created 12 test cases for evaluation.
   1. What is the return policy for regular items?...
   2. How long do I have to return electronics?...
   3. What shipping options are available and how much do they cos...
   4. Do you ship internationally?...
   5. What does the warranty cover on Acme products?...
   6. How can I contact customer support?...
   7. What are the features of the Acme SmartHome Hub?...
   8. How much is the AirPure Pro and what does it filter?...
   9. What payment methods do you accept?...
  10. How does the Acme Rewards program work?...
  11. Can I cancel an order after placing it?...
  12. Tell me about the Acme ErgoDesk Pro standing desk....


-
## 13. Run Pipeline on All Test Cases

We run the RAG pipeline on all test cases and store the results for use in evaluation notebooks.

In [16]:
pipeline_results = []

for i, tc in enumerate(TEST_CASES, 1):
    print(f"Processing test case {i}/{len(TEST_CASES)}: {tc['query'][:50]}...")
    answer, contexts = rag_pipeline(tc["query"], verbose=False)
    
    pipeline_results.append({
        "query": tc["query"],
        "expected_output": tc["expected_output"],
        "actual_output": answer,
        "retrieval_context": contexts,
    })

print(f"\nAll {len(pipeline_results)} test cases processed.")

Processing test case 1/12: What is the return policy for regular items?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 2/12: How long do I have to return electronics?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 3/12: What shipping options are available and how much d...
  [Cross-encoder reranker applied — 3 results]
Processing test case 4/12: Do you ship internationally?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 5/12: What does the warranty cover on Acme products?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 6/12: How can I contact customer support?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 7/12: What are the features of the Acme SmartHome Hub?...
  [Cross-encoder reranker applied — 3 results]
Processing test case 8/12: How much is the AirPure Pro and what does it filte...
  [Cross-encoder reranker applied — 3 results]
Processing test case 9/12: What pa

In [17]:
# Display results summary
import pandas as pd

summary_df = pd.DataFrame([
    {
        "Query": r["query"][:50] + "...",
        "Answer Length": len(r["actual_output"]),
        "Num Contexts": len(r["retrieval_context"]),
    }
    for r in pipeline_results
])

print(summary_df.to_string(index=False))

                                                Query  Answer Length  Num Contexts
      What is the return policy for regular items?...            554             3
         How long do I have to return electronics?...             73             3
What shipping options are available and how much d...            724             3
                      Do you ship internationally?...             58             3
    What does the warranty cover on Acme products?...            430             3
               How can I contact customer support?...            543             3
  What are the features of the Acme SmartHome Hub?...            469             3
How much is the AirPure Pro and what does it filte...            162             3
               What payment methods do you accept?...            377             3
           How does the Acme Rewards program work?...            459             3
           Can I cancel an order after placing it?...            341             3
Tell

-
## 14. Save Pipeline Results for Other Notebooks

We save the test case results to a JSON file so that later notebooks can load them directly instead of re-running the pipeline. We also save the knowledge base and test cases.

In [18]:
import json

output_dir = os.path.join(os.getcwd(), "data")
os.makedirs(output_dir, exist_ok=True)

# Save pipeline results
with open(os.path.join(output_dir, "pipeline_results.json"), "w") as f:
    json.dump(pipeline_results, f, indent=2)

# Save test cases
with open(os.path.join(output_dir, "test_cases.json"), "w") as f:
    json.dump(TEST_CASES, f, indent=2)

# Save knowledge base
kb_data = [{"title": title, "text": text} for title, text in KNOWLEDGE_BASE]
with open(os.path.join(output_dir, "knowledge_base.json"), "w") as f:
    json.dump(kb_data, f, indent=2)

print(f"Saved to {output_dir}/:")
print(f"  - pipeline_results.json  ({len(pipeline_results)} results)")
print(f"  - test_cases.json        ({len(TEST_CASES)} test cases)")
print(f"  - knowledge_base.json    ({len(kb_data)} documents)")

Saved to /Users/fnu.satvik/dev/github.com/satvik/ragevals/workbooks/data/:
  - pipeline_results.json  (12 results)
  - test_cases.json        (12 test cases)
  - knowledge_base.json    (15 documents)


-
## 15. How to Use in Later Notebooks

In Notebooks 03-05, you can load the saved results like this:

```python
import json

with open("data/pipeline_results.json") as f:
    pipeline_results = json.load(f)

# Each result has: query, expected_output, actual_output, retrieval_context
```

Alternatively, you can rebuild the pipeline inline by copying the key functions from this notebook. The later notebooks include both options.

-
## 16. Pipeline Architecture Summary

```
User Query
    |
    v
[OpenAI Embeddings] -- embed query
    |
    v
[Qdrant Vector Search] -- retrieve top-K chunks
    |
    v
[Cross-Encoder Reranker] -- rerank to top-N (optional, runs locally)
    |
    v
[GPT-4o-mini Generator] -- generate answer from contexts
    |
    v
(answer, contexts) --> Ready for evaluation
```

**Key Design Decisions:**

- **Embedding model:** `text-embedding-3-small` (1536 dim, good cost-quality tradeoff)
- **Vector DB:** Qdrant in-memory (no external server needed for this tutorial)
- **Chunking:** ~400 chars with 50-char overlap (sentence-boundary aware)
- **Retrieval:** Top-5 by cosine similarity
- **Reranking:** `cross-encoder/ms-marco-MiniLM-L-6-v2` to top-3 (optional, runs locally — no API key needed)
- **Generation:** GPT-4o-mini with temperature 0.1 for consistent, grounded answers

-
## Next Steps

The RAG pipeline is complete and tested. Proceed to:

- **Notebook 03** — Evaluate retriever quality (Contextual Relevancy, Precision, Recall)
- **Notebook 04** — Evaluate generator quality (Answer Relevancy, Faithfulness, Hallucination)
- **Notebook 05** — Advanced evaluation (G-Eval, custom metrics, synthetic data)